In [2]:
#https://github.com/DataTalksClub/data-engineering-zoomcamp/blob/main/03-data-warehouse/extras/web_to_gcs.py

import os
import requests
import pandas as pd
from google.cloud import storage
from dotenv import load_dotenv, find_dotenv

root_dir = "/workspaces/finance-data-pipeline/"

In [5]:
"""
Pre-reqs: 
1. run `uv sync` from this 'extra' folder (create venv and install dependencies from pyproject.toml)
2. rename .env-example to .env (not commited thanks to .gitignore)
3. in .env, 
    - set GCP_GCS_BUCKET as your bucket or change default value of BUCKET
    - Set GOOGLE_APPLICATION_CREDENTIALS to your project/service-account json key 
    (or don't set it if you use google ADC)
"""
root = os.path.dirname(os.getcwd())
print(f"Current working directory: {root}")
# load env vars from .env
load_dotenv(dotenv_path=f"{root}/.env/.env")
load_dotenv()

python-dotenv could not parse statement starting at line 1
python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 3


Current working directory: /workspaces/finance-data-pipeline


False

In [ ]:
BUCKET = os.environ.get("GCP_GCS_BUCKET", "findata-bucket")
project_id = os.environ.get("GOOGLE_CLOUD_PROJECT")

Using project ID: None


In [45]:
import glob
from pathlib import Path

# Define data directories
raw_data_dir = "/workspaces/finance-data-pipeline/raw_data/"
database_dir = "/workspaces/finance-data-pipeline/database/"

# List available files
raw_files = glob.glob(f"{raw_data_dir}*")
db_files = glob.glob(f"{database_dir}*.duckdb")

print("=== Available Raw Data Files ===")
for file in sorted(raw_files):
    file_size = Path(file).stat().st_size / (1024 * 1024)  # Convert to MB
    print(f"  • {Path(file).name} ({file_size:.2f} MB)")

print("\n=== Available Database Files ===")
for file in sorted(db_files):
    file_size = Path(file).stat().st_size / (1024 * 1024)  # Convert to MB
    print(f"  • {Path(file).name} ({file_size:.2f} MB)")

=== Available Raw Data Files ===
  • balance_sheet.csv (0.01 MB)
  • cash_flow_statement.csv (0.01 MB)
  • company_info.csv (0.01 MB)
  • financials.json (0.17 MB)
  • income_statement.csv (0.01 MB)

=== Available Database Files ===
  • finance.duckdb (3.26 MB)


## List Available Files

Explore which files are ready to upload

### Function definitions

In [46]:
def upload_to_gcs(bucket_name, object_name, local_file, project_id=None):
    """
    Upload a file to Google Cloud Storage.
    
    Args:
        bucket_name (str): GCS bucket name
        object_name (str): The path/name in GCS (e.g., "raw_data/financials.json")
        local_file (str): Local file path to upload
    
    Returns:
        bool: True if successful, False otherwise
    """
    try:
        client = storage.Client(project=project_id)
        bucket = client.bucket(bucket_name)
        blob = bucket.blob(object_name)
        
        print(f"Uploading {local_file} to gs://{bucket_name}/{object_name}...", end=" ")
        blob.upload_from_filename(local_file)
        print("✓ Done")
        return True
    except Exception as e:
        print(f"✗ Failed: {str(e)}")
        return False

In [47]:
def upload_raw_data_files(bucket_name=BUCKET, raw_data_path=raw_data_dir, project_id=None):
    """
    Upload all raw data files (CSV and JSON) to GCS.
    
    Args:
        bucket_name (str): GCS bucket name
        raw_data_path (str): Path to raw data directory
        project_id (str): GCP project ID
    
    Returns:
        dict: Summary of uploads with success/failure counts
    """
    raw_files = glob.glob(f"{raw_data_path}*")
    
    success_count = 0
    failed_count = 0
    results = []
    
    print("=" * 60)
    print("UPLOADING RAW DATA FILES")
    print("=" * 60)
    
    for local_file in sorted(raw_files):
        file_name = Path(local_file).name
        
        # Skip non-data files
        if file_name.startswith("."):
            continue
        
        # Create GCS path
        gcs_path = f"raw_data/{file_name}"
        
        # Upload file
        if upload_to_gcs(bucket_name, gcs_path, local_file, project_id=project_id):
            success_count += 1
            results.append({"file": file_name, "status": "✓ Success"})
        else:
            failed_count += 1
            results.append({"file": file_name, "status": "✗ Failed"})
    
    print("\n" + "=" * 60)
    print(f"Summary: {success_count} succeeded, {failed_count} failed")
    print("=" * 60)
    
    return {"success": success_count, "failed": failed_count, "results": results}

In [48]:
# def upload_database_files(bucket_name=BUCKET, database_path=database_dir):
#     """
#     Upload database files (DuckDB) to GCS.
    
#     Args:
#         bucket_name (str): GCS bucket name
#         database_path (str): Path to database directory
    
#     Returns:
#         dict: Summary of uploads
#     """
#     db_files = glob.glob(f"{database_path}*.duckdb")
    
#     success_count = 0
#     failed_count = 0
    
#     print("=" * 60)
#     print("UPLOADING DATABASE FILES")
#     print("=" * 60)
    
#     for local_file in sorted(db_files):
#         file_name = Path(local_file).name
        
#         # Create GCS path
#         gcs_path = f"database/{file_name}"
        
#         # Upload file
#         if upload_to_gcs(bucket_name, gcs_path, local_file):
#             success_count += 1
#         else:
#             failed_count += 1
    
#     print("\n" + "=" * 60)
#     print(f"Summary: {success_count} succeeded, {failed_count} failed")
#     print("=" * 60)
    
#     return {"success": success_count, "failed": failed_count}

In [49]:
def list_gcs_files(bucket_name=BUCKET, prefix="", project_id=None):
    """
    List all files in GCS bucket with optional prefix filter.
    
    Args:
        bucket_name (str): GCS bucket name
        prefix (str): Optional prefix to filter files (e.g., "raw_data/")
        project_id (str): GCP project ID
    
    Returns:
        list: List of file names in GCS
    """
    try:
        client = storage.Client(project=project_id)
        bucket = client.bucket(bucket_name)
        blobs = bucket.list_blobs(prefix=prefix)
        
        files = []
        for blob in blobs:
            files.append({
                "name": blob.name,
                "size_mb": blob.size / (1024 * 1024),
                "updated": blob.updated
            })
        
        return files
    except Exception as e:
        print(f"Error listing files: {str(e)}")
        return []

def verify_uploads(bucket_name=BUCKET, project_id=None):
    """
    Verify all uploaded files in GCS.
    """
    print("=" * 60)
    print("VERIFYING UPLOADS IN GCS")
    print("=" * 60)
    
    print("\n📁 Raw Data Files:")
    raw_files = list_gcs_files(bucket_name, "raw_data/", project_id=project_id)
    if raw_files:
        for f in raw_files:
            print(f"  • {Path(f['name']).name} ({f['size_mb']:.2f} MB)")
    else:
        print("  No files found")
    
    print("\n💾 Database Files:")
    db_files = list_gcs_files(bucket_name, "database/", project_id=project_id)
    if db_files:
        for f in db_files:
            print(f"  • {Path(f['name']).name} ({f['size_mb']:.2f} MB)")
    else:
        print("  No files found")
    
    total_files = len(raw_files) + len(db_files)
    print("\n" + "=" * 60)
    print(f"Total files in GCS: {total_files}")
    print("=" * 60)

In [50]:
def upload_specific_file(file_name, file_type="raw_data", bucket_name=BUCKET, project_id=None):
    """
    Upload a specific file to GCS.
    
    Args:
        file_name (str): Name of the file to upload
        file_type (str): Type - 'raw_data' or 'database'
        bucket_name (str): GCS bucket name
        project_id (str): GCP project ID
    
    Returns:
        bool: True if successful
    """
    if file_type == "raw_data":
        local_file = f"{raw_data_dir}{file_name}"
    elif file_type == "database":
        local_file = f"{database_dir}{file_name}"
    else:
        print(f"Unknown file type: {file_type}")
        return False
    
    if not Path(local_file).exists():
        print(f"File not found: {local_file}")
        return False
    
    gcs_path = f"{file_type}/{file_name}"
    return upload_to_gcs(bucket_name, gcs_path, local_file, project_id=project_id)

# Example: Upload specific files
# upload_specific_file("financials.json", "raw_data", project_id=project_id)
# upload_specific_file("finance.duckdb", "database", project_id=project_id)

In [51]:
def get_upload_statistics(bucket_name=BUCKET, project_id=None):
    """
    Generate statistics about local and uploaded files.
    """
    print("=" * 60)
    print("UPLOAD STATISTICS")
    print("=" * 60)
    
    # Local files statistics
    print("\n📊 LOCAL FILES:")
    raw_files = glob.glob(f"{raw_data_dir}*")
    db_files = glob.glob(f"{database_dir}*.duckdb")
    
    raw_data_size = sum(Path(f).stat().st_size for f in raw_files if not Path(f).name.startswith("."))
    db_data_size = sum(Path(f).stat().st_size for f in db_files)
    
    print(f"  Raw Data: {len(raw_files)} files, {raw_data_size / (1024 * 1024):.2f} MB")
    print(f"  Database: {len(db_files)} files, {db_data_size / (1024 * 1024):.2f} MB")
    print(f"  Total: {raw_data_size + db_data_size / (1024 * 1024):.2f} MB")
    
    # GCS files statistics
    print("\n☁️  GCS FILES:")
    try:
        all_gcs_files = list_gcs_files(bucket_name, project_id=project_id)
        gcs_size = sum(f['size_mb'] for f in all_gcs_files)
        print(f"  Total uploaded: {len(all_gcs_files)} files, {gcs_size:.2f} MB")
    except Exception as e:
        print(f"  Error retrieving GCS stats: {str(e)}")
    
    print("\n" + "=" * 60)

# Get statistics
get_upload_statistics(project_id=project_id)

/workspaces/finance-data-pipeline/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


UPLOAD STATISTICS

📊 LOCAL FILES:
  Raw Data: 5 files, 0.20 MB
  Database: 1 files, 3.26 MB
  Total: 209167.26 MB

☁️  GCS FILES:
  Total uploaded: 1 files, 0.17 MB



### Executing functions

In [41]:

upload_to_gcs(BUCKET, 
              "raw_data/financials.json", 
              "/workspaces/finance-data-pipeline/raw_data/financials.json", 
              project_id=project_id)

/workspaces/finance-data-pipeline/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Uploading /workspaces/finance-data-pipeline/raw_data/financials.json to gs://findata-bucket/raw_data/financials.json... ✓ Done


True

In [ ]:
upload_to_gcs(BUCKET, \
            "raw_data/financials.json", \
            "/workspaces/finance-data-pipeline/raw_data/financials.json", \
            project_id=project_id)

In [ ]:
import os
root_dir = os.path.dirname(os.getcwd())
root_dir

with open(root_dir + "/raw_data/data.txt", "w") as f:
    data = f.write("Hello, World!")


# from pathlib import Path
# root_dir = Path.cwd().parent 

In [ ]:
verify_uploads(bucket_name=BUCKET, project_id=project_id)

VERIFYING UPLOADS IN GCS

📁 Raw Data Files:
Error listing files: Project was not passed and could not be determined from the environment.
  No files found

💾 Database Files:
Error listing files: Project was not passed and could not be determined from the environment.
  No files found

Total files in GCS: 0


/workspaces/finance-data-pipeline/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)
/workspaces/finance-data-pipeline/.venv/lib/python3.13/site-packages/google/auth/_default.py:114: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


In [ ]:
# Run the complete upload workflow
raw_data_results = upload_raw_data_files(project_id=project_id)
# db_results = upload_database_files(project_id=project_id)

# Verify uploads
print("\n")
verify_uploads(project_id=project_id)

In [ ]:
"""
USAGE GUIDE - Common Operations:

1. Upload all raw data files:
   >>> raw_data_results = upload_raw_data_files(project_id=project_id)

2. Upload all database files:
   >>> db_results = upload_database_files(project_id=project_id)

3. Upload a specific file:
   >>> upload_specific_file("financials.json", "raw_data", project_id=project_id)
   >>> upload_specific_file("finance.duckdb", "database", project_id=project_id)

4. List all files in GCS:
   >>> all_files = list_gcs_files(prefix="", project_id=project_id)

5. List only raw data files in GCS:
   >>> raw_files = list_gcs_files(prefix="raw_data/", project_id=project_id)

6. Verify all uploads:
   >>> verify_uploads(project_id=project_id)

7. Get statistics:
   >>> get_upload_statistics(project_id=project_id)

REQUIREMENTS:
- .env file with GCP_GCS_BUCKET and GOOGLE_APPLICATION_CREDENTIALS set
- .env file with GOOGLE_CLOUD_PROJECT set for proper GCP authentication
- google-cloud-storage package installed
- GCP authentication configured (ADC or service account key)
"""

print(__doc__)

## Usage Guide & Examples

Common use cases and examples for uploading to GCP

## Upload Statistics & Monitoring

View upload summary and data statistics

## Selective File Upload

Upload specific files to GCS

## Execute Upload Process

Run the complete upload workflow (raw data + database files)

## Verify Uploaded Files

Check what files exist in GCS

## Upload Database Files

Upload DuckDB database files to GCS

## Upload Raw Data Files

Upload CSV and JSON files from raw_data directory